In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np
from torch.utils.data import DataLoader, Dataset

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

class AlbaniaSATDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), \
               torch.tensor(self.y[idx], dtype=torch.long)

class_names = [
    "Broad-leaved Forest", "Coniferous Forest", "Shrubland",
    "Agricultural", "Grassland", "Olive Groves", "Urban", "Water"
]

eurosat_classes = [
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway",
    "Industrial", "Pasture", "PermanentCrop", "Residential",
    "River", "SeaLake"
]

Using device: mps


In [2]:
X = np.load("../data/AlbaniaSAT/processed_v2/patches.npy")
y = np.load("../data/AlbaniaSAT/processed_v2/labels.npy")

# Split — same seed as training
n = len(X)
indices = np.random.RandomState(42).permutation(n)
n_train = int(0.7 * n)
n_val   = int(0.15 * n)
test_idx = indices[n_train + n_val:]

# RGB test set (3 bands)
X_rgb = X[:, :3, :, :]
X_rgb = np.clip(X_rgb, 0, 3000) / 3000.0
mean_3 = np.array([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1)
std_3  = np.array([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1)
X_rgb = (X_rgb - mean_3) / std_3
X_rgb = X_rgb.astype(np.float32)

# 4-band test set
X_4band = X[:, :4, :, :]
X_4band = np.clip(X_4band, 0, 3000) / 3000.0
mean_4 = np.array([0.485, 0.456, 0.406, 0.441]).reshape(1, 4, 1, 1)
std_4  = np.array([0.229, 0.224, 0.225, 0.220]).reshape(1, 4, 1, 1)
X_4band = (X_4band - mean_4) / std_4
X_4band = X_4band.astype(np.float32)

# Test loaders
test_rgb  = DataLoader(AlbaniaSATDataset(X_rgb[test_idx],   y[test_idx]), batch_size=32)
test_4band = DataLoader(AlbaniaSATDataset(X_4band[test_idx], y[test_idx]), batch_size=32)

print(f"Test set size: {len(test_idx)} patches")

Test set size: 1200 patches


In [3]:
def evaluate(model, loader, model_name):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            preds = outputs.argmax(1)
            correct += (preds == y_batch).sum().item()
            total += len(y_batch)
    acc = correct / total * 100
    print(f"{model_name}: {acc:.2f}%")
    return acc

In [4]:
results = {}

# Model 1 — EuroSAT ResNet50 zero-shot
eurosat_ckpt = torch.load("../results/models/resnet50_eurosat.pth", map_location="cpu")
model_eurosat = models.resnet50()
model_eurosat.fc = nn.Linear(2048, 10)
model_eurosat.load_state_dict(eurosat_ckpt)
model_eurosat = model_eurosat.to(device)

# Map AlbaniaSAT labels to closest EuroSAT equivalent
mapping = {0:1, 1:1, 2:2, 3:0, 4:5, 5:6, 6:7, 7:9}
y_test = y[test_idx]
y_eurosat_equiv = np.array([mapping[label] for label in y_test])

model_eurosat.eval()
all_preds = []
with torch.no_grad():
    for X_batch, _ in test_rgb:
        X_batch = X_batch.to(device)
        preds = model_eurosat(X_batch).argmax(1).cpu().numpy()
        all_preds.extend(preds)
all_preds = np.array(all_preds)
acc_eurosat = (all_preds == y_eurosat_equiv).mean() * 100
print(f"EuroSAT zero-shot: {acc_eurosat:.2f}%")
results["EuroSAT zero-shot (RGB)"] = acc_eurosat

# Model 2 — ResNet50 finetuned on AlbaniaSAT v2 RGB
model_ft = models.resnet50()
model_ft.fc = nn.Linear(2048, 8)
model_ft.load_state_dict(torch.load("../results/models/resnet50_albaniasat_v2.pth", map_location="cpu"))
model_ft = model_ft.to(device)
results["ResNet50 finetuned (RGB)"] = evaluate(model_ft, test_rgb, "ResNet50 finetuned (RGB)")

# Model 3 — SoftCon finetuned on AlbaniaSAT
from huggingface_hub import hf_hub_download
weights_path = hf_hub_download(repo_id="wangyi111/softcon", filename="B13_rn50_softcon.pth")

model_softcon = models.resnet50()
state_dict = torch.load(weights_path, map_location="cpu")
if "model" in state_dict:
    state_dict = state_dict["model"]
conv1_weight_4 = state_dict["conv1.weight"][:, :4, :, :]
state_dict["conv1.weight"] = conv1_weight_4
model_softcon.conv1 = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)
model_softcon.fc = nn.Linear(2048, 8)
model_softcon.load_state_dict(
    torch.load("../results/models/resnet50_softcon_albaniasat.pth", map_location="cpu"),
    strict=False
)
model_softcon = model_softcon.to(device)
results["SoftCon finetuned (4-band)"] = evaluate(model_softcon, test_4band, "SoftCon finetuned (4-band)")

# Summary
print("\n" + "=" * 45)
print("FINAL RESULTS SUMMARY")
print("=" * 45)
for model_name, acc in results.items():
    print(f"{model_name:<35} {acc:.2f}%")

EuroSAT zero-shot: 19.08%
ResNet50 finetuned (RGB): 64.08%
SoftCon finetuned (4-band): 64.33%

FINAL RESULTS SUMMARY
EuroSAT zero-shot (RGB)             19.08%
ResNet50 finetuned (RGB)            64.08%
SoftCon finetuned (4-band)          64.33%


In [6]:
from torch.utils.data import Dataset

class AlbaniaSATDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = X
        self.y = y
        self.augment = augment

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx], dtype=torch.float32)
        y = torch.tensor(self.y[idx], dtype=torch.long)
        if self.augment:
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[2])
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[1])
            k = torch.randint(0, 4, (1,)).item()
            x = torch.rot90(x, k, dims=[1, 2])
        return x, y

In [7]:
# Rebuild 4-band dataloader with 8000 patches
X = np.load("../data/AlbaniaSAT/processed_v2/patches.npy")
y = np.load("../data/AlbaniaSAT/processed_v2/labels.npy")

X_4band = X[:, :4, :, :]
mean_4 = np.array([0.485, 0.456, 0.406, 0.441]).reshape(1, 4, 1, 1)
std_4  = np.array([0.229, 0.224, 0.225, 0.220]).reshape(1, 4, 1, 1)
X_4band = np.clip(X_4band, 0, 3000) / 3000.0
X_4band = (X_4band - mean_4) / std_4
X_4band = X_4band.astype(np.float32)

n = len(X_4band)
indices = np.random.RandomState(42).permutation(n)
n_train = int(0.7 * n)
n_val   = int(0.15 * n)

train_idx = indices[:n_train]
val_idx   = indices[n_train:n_train + n_val]
test_idx  = indices[n_train + n_val:]

train_set = AlbaniaSATDataset(X_4band[train_idx], y[train_idx], augment=True)
val_set   = AlbaniaSATDataset(X_4band[val_idx],   y[val_idx],   augment=False)
test_set  = AlbaniaSATDataset(X_4band[test_idx],  y[test_idx],  augment=False)

train_loader_4 = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader_4   = DataLoader(val_set,   batch_size=32, shuffle=False)
test_loader_4  = DataLoader(test_set,  batch_size=32, shuffle=False)

print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

Train: 5600 | Val: 1200 | Test: 1200


In [8]:
def train_stage(model, train_loader, val_loader, epochs, lr, stage_name):
    model = model.to(device)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    best_val_acc = 0
    best_weights = None

    for epoch in range(epochs):
        model.train()
        train_correct = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_correct += (outputs.argmax(1) == y_batch).sum().item()

        model.eval()
        val_correct = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                val_correct += (outputs.argmax(1) == y_batch).sum().item()

        train_acc = train_correct / len(train_loader.dataset) * 100
        val_acc   = val_correct   / len(val_loader.dataset)   * 100

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        scheduler.step()
        print(f"{stage_name} | Epoch {epoch+1}/{epochs} | "
              f"Train: {train_acc:.2f}% | Val: {val_acc:.2f}%")

    model.load_state_dict(best_weights)
    model = model.to(device)
    print(f"\n{stage_name} done! Best val accuracy: {best_val_acc:.2f}%\n")
    return model

In [9]:
from huggingface_hub import hf_hub_download

weights_path = hf_hub_download(repo_id="wangyi111/softcon", filename="B13_rn50_softcon.pth")

# Load fresh SoftCon
model_softcon_v2 = models.resnet50()
state_dict = torch.load(weights_path, map_location="cpu")
if "model" in state_dict:
    state_dict = state_dict["model"]

conv1_weight_4 = state_dict["conv1.weight"][:, :4, :, :]
state_dict["conv1.weight"] = conv1_weight_4
model_softcon_v2.conv1 = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)
model_softcon_v2.fc = nn.Identity()
model_softcon_v2.load_state_dict(state_dict, strict=False)

# Freeze everything
for param in model_softcon_v2.parameters():
    param.requires_grad = False

# Replace classifier
model_softcon_v2.fc = nn.Linear(2048, 8)
model_softcon_v2 = model_softcon_v2.to(device)

print("=" * 50)
print("STAGE 1 — Training classifier head only")
print("=" * 50)
model_softcon_v2 = train_stage(model_softcon_v2, train_loader_4, val_loader_4,
                    epochs=5, lr=1e-3, stage_name="Stage 1")

print("=" * 50)
print("STAGE 2 — Unfreezing layer4")
print("=" * 50)
for param in model_softcon_v2.layer4.parameters():
    param.requires_grad = True
model_softcon_v2 = train_stage(model_softcon_v2, train_loader_4, val_loader_4,
                    epochs=5, lr=1e-4, stage_name="Stage 2")

print("=" * 50)
print("STAGE 3 — Unfreezing layer3")
print("=" * 50)
for param in model_softcon_v2.layer3.parameters():
    param.requires_grad = True
model_softcon_v2 = train_stage(model_softcon_v2, train_loader_4, val_loader_4,
                    epochs=5, lr=1e-5, stage_name="Stage 3")

torch.save(model_softcon_v2.state_dict(), "../results/models/resnet50_softcon_albaniasat_v2.pth")
print("SoftCon v2 saved!")

STAGE 1 — Training classifier head only
Stage 1 | Epoch 1/5 | Train: 51.89% | Val: 57.58%
Stage 1 | Epoch 2/5 | Train: 58.41% | Val: 58.42%
Stage 1 | Epoch 3/5 | Train: 58.86% | Val: 58.75%
Stage 1 | Epoch 4/5 | Train: 59.54% | Val: 58.75%
Stage 1 | Epoch 5/5 | Train: 59.55% | Val: 59.08%

Stage 1 done! Best val accuracy: 59.08%

STAGE 2 — Unfreezing layer4
Stage 2 | Epoch 1/5 | Train: 61.30% | Val: 61.83%
Stage 2 | Epoch 2/5 | Train: 64.93% | Val: 61.92%
Stage 2 | Epoch 3/5 | Train: 68.41% | Val: 63.00%
Stage 2 | Epoch 4/5 | Train: 70.88% | Val: 63.25%
Stage 2 | Epoch 5/5 | Train: 71.93% | Val: 63.25%

Stage 2 done! Best val accuracy: 63.25%

STAGE 3 — Unfreezing layer3
Stage 3 | Epoch 1/5 | Train: 72.50% | Val: 63.25%
Stage 3 | Epoch 2/5 | Train: 73.14% | Val: 63.17%
Stage 3 | Epoch 3/5 | Train: 74.04% | Val: 64.50%
Stage 3 | Epoch 4/5 | Train: 74.05% | Val: 63.58%
Stage 3 | Epoch 5/5 | Train: 75.50% | Val: 64.00%

Stage 3 done! Best val accuracy: 64.50%

SoftCon v2 saved!


In [11]:
from huggingface_hub import hf_hub_download

weights_path = hf_hub_download(repo_id="wangyi111/softcon", filename="B13_rn50_softcon.pth")

# Load fresh SoftCon
model_softcon_v3 = models.resnet50()
state_dict = torch.load(weights_path, map_location="cpu")
if "model" in state_dict:
    state_dict = state_dict["model"]

conv1_weight_4 = state_dict["conv1.weight"][:, :4, :, :]
state_dict["conv1.weight"] = conv1_weight_4
model_softcon_v3.conv1 = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)
model_softcon_v3.fc = nn.Identity()
model_softcon_v3.load_state_dict(state_dict, strict=False)

# Freeze everything
for param in model_softcon_v3.parameters():
    param.requires_grad = False

# Replace classifier
model_softcon_v3.fc = nn.Linear(2048, 8)
model_softcon_v3 = model_softcon_v3.to(device)

print("=" * 50)
print("STAGE 1 — Training classifier head only")
print("=" * 50)
model_softcon_v3 = train_stage(model_softcon_v3, train_loader_4, val_loader_4,
                    epochs=5, lr=1e-3, stage_name="Stage 1")

print("=" * 50)
print("STAGE 2 — Unfreezing layer4 (10 epochs)")
print("=" * 50)
for param in model_softcon_v3.layer4.parameters():
    param.requires_grad = True
model_softcon_v3 = train_stage(model_softcon_v3, train_loader_4, val_loader_4,
                    epochs=10, lr=1e-4, stage_name="Stage 2")

print("=" * 50)
print("STAGE 3 — Unfreezing layer3 (10 epochs)")
print("=" * 50)
for param in model_softcon_v3.layer3.parameters():
    param.requires_grad = True
model_softcon_v3 = train_stage(model_softcon_v3, train_loader_4, val_loader_4,
                    epochs=10, lr=1e-5, stage_name="Stage 3")

torch.save(model_softcon_v3.state_dict(), "../results/models/resnet50_softcon_albaniasat_v3.pth")
print("SoftCon v3 saved!")

STAGE 1 — Training classifier head only
Stage 1 | Epoch 1/5 | Train: 52.59% | Val: 56.75%
Stage 1 | Epoch 2/5 | Train: 57.63% | Val: 58.58%
Stage 1 | Epoch 3/5 | Train: 58.93% | Val: 57.92%
Stage 1 | Epoch 4/5 | Train: 59.39% | Val: 58.92%
Stage 1 | Epoch 5/5 | Train: 59.89% | Val: 58.92%

Stage 1 done! Best val accuracy: 58.92%

STAGE 2 — Unfreezing layer4 (10 epochs)
Stage 2 | Epoch 1/10 | Train: 60.93% | Val: 62.00%
Stage 2 | Epoch 2/10 | Train: 64.16% | Val: 64.33%
Stage 2 | Epoch 3/10 | Train: 67.21% | Val: 63.08%
Stage 2 | Epoch 4/10 | Train: 69.73% | Val: 62.58%
Stage 2 | Epoch 5/10 | Train: 72.34% | Val: 61.50%
Stage 2 | Epoch 6/10 | Train: 74.41% | Val: 62.58%
Stage 2 | Epoch 7/10 | Train: 76.79% | Val: 63.83%
Stage 2 | Epoch 8/10 | Train: 78.70% | Val: 63.17%
Stage 2 | Epoch 9/10 | Train: 80.21% | Val: 63.42%
Stage 2 | Epoch 10/10 | Train: 81.09% | Val: 62.83%

Stage 2 done! Best val accuracy: 64.33%

STAGE 3 — Unfreezing layer3 (10 epochs)
Stage 3 | Epoch 1/10 | Train: 68.50

In [12]:
# ResNet50 finetuned on 8000 patches
checkpoint = torch.load("../results/models/resnet50_eurosat.pth", map_location="cpu")
model_resnet_v2 = models.resnet50()
model_resnet_v2.fc = nn.Linear(2048, 10)
model_resnet_v2.load_state_dict(checkpoint)

# Freeze everything
for param in model_resnet_v2.parameters():
    param.requires_grad = False

# Replace classifier
model_resnet_v2.fc = nn.Linear(2048, 8)
model_resnet_v2 = model_resnet_v2.to(device)

# Rebuild RGB dataloader with 8000 patches
X = np.load("../data/AlbaniaSAT/processed_v2/patches.npy")
y = np.load("../data/AlbaniaSAT/processed_v2/labels.npy")

X_rgb = X[:, :3, :, :]
X_rgb = np.clip(X_rgb, 0, 3000) / 3000.0
mean_3 = np.array([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1)
std_3  = np.array([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1)
X_rgb = (X_rgb - mean_3) / std_3
X_rgb = X_rgb.astype(np.float32)

n = len(X_rgb)
indices = np.random.RandomState(42).permutation(n)
n_train = int(0.7 * n)
n_val   = int(0.15 * n)

train_idx = indices[:n_train]
val_idx   = indices[n_train:n_train + n_val]
test_idx  = indices[n_train + n_val:]

train_set_rgb = AlbaniaSATDataset(X_rgb[train_idx], y[train_idx], augment=True)
val_set_rgb   = AlbaniaSATDataset(X_rgb[val_idx],   y[val_idx],   augment=False)
test_set_rgb  = AlbaniaSATDataset(X_rgb[test_idx],  y[test_idx],  augment=False)

train_loader_rgb = DataLoader(train_set_rgb, batch_size=32, shuffle=True)
val_loader_rgb   = DataLoader(val_set_rgb,   batch_size=32, shuffle=False)
test_loader_rgb  = DataLoader(test_set_rgb,  batch_size=32, shuffle=False)

print(f"Train: {len(train_set_rgb)} | Val: {len(val_set_rgb)} | Test: {len(test_set_rgb)}")

print("=" * 50)
print("STAGE 1 — Training classifier head only")
print("=" * 50)
model_resnet_v2 = train_stage(model_resnet_v2, train_loader_rgb, val_loader_rgb,
                    epochs=5, lr=1e-3, stage_name="Stage 1")

print("=" * 50)
print("STAGE 2 — Unfreezing layer4 (10 epochs)")
print("=" * 50)
for param in model_resnet_v2.layer4.parameters():
    param.requires_grad = True
model_resnet_v2 = train_stage(model_resnet_v2, train_loader_rgb, val_loader_rgb,
                    epochs=10, lr=1e-4, stage_name="Stage 2")

print("=" * 50)
print("STAGE 3 — Unfreezing layer3 (10 epochs)")
print("=" * 50)
for param in model_resnet_v2.layer3.parameters():
    param.requires_grad = True
model_resnet_v2 = train_stage(model_resnet_v2, train_loader_rgb, val_loader_rgb,
                    epochs=10, lr=1e-5, stage_name="Stage 3")

torch.save(model_resnet_v2.state_dict(), "../results/models/resnet50_albaniasat_v3.pth")
print("ResNet v3 saved!")

Train: 5600 | Val: 1200 | Test: 1200
STAGE 1 — Training classifier head only
Stage 1 | Epoch 1/5 | Train: 44.75% | Val: 52.42%
Stage 1 | Epoch 2/5 | Train: 51.48% | Val: 52.67%
Stage 1 | Epoch 3/5 | Train: 54.61% | Val: 55.17%
Stage 1 | Epoch 4/5 | Train: 54.91% | Val: 54.08%
Stage 1 | Epoch 5/5 | Train: 55.73% | Val: 55.92%

Stage 1 done! Best val accuracy: 55.92%

STAGE 2 — Unfreezing layer4 (10 epochs)
Stage 2 | Epoch 1/10 | Train: 57.20% | Val: 59.33%
Stage 2 | Epoch 2/10 | Train: 61.59% | Val: 62.00%
Stage 2 | Epoch 3/10 | Train: 63.75% | Val: 62.92%
Stage 2 | Epoch 4/10 | Train: 65.61% | Val: 63.00%
Stage 2 | Epoch 5/10 | Train: 66.71% | Val: 63.50%
Stage 2 | Epoch 6/10 | Train: 68.36% | Val: 63.75%
Stage 2 | Epoch 7/10 | Train: 70.09% | Val: 64.00%
Stage 2 | Epoch 8/10 | Train: 71.16% | Val: 63.75%
Stage 2 | Epoch 9/10 | Train: 72.29% | Val: 64.25%
Stage 2 | Epoch 10/10 | Train: 72.18% | Val: 63.92%

Stage 2 done! Best val accuracy: 64.25%

STAGE 3 — Unfreezing layer3 (10 epochs